In [ ]:
import requests
import json
from notebookutils import mssparkutils

# --- Configuration ---
# API Base URL
fabric_api_base = "https://api.fabric.microsoft.com/v1"

# Auth Details (Update with your SPN or Key Vault logic)
tenant_id = "your-tenant-id"
client_id = "your-client-id"
client_secret = "your-client-secret" 
# Recommended: Use Key Vault
# client_secret = mssparkutils.credentials.getSecret("your-kv", "your-secret")

# --- Parameters ---
# Replace with actual IDs you want to query
workspace_id = "your-workspace-id"
item_id = "your-item-id"

In [ ]:
def get_spn_token(scope):
    url = f"https://login.microsoftonline.com/{tenant_id}/oauth2/v2.0/token"
    payload = {
        'grant_type': 'client_credentials',
        'client_id': client_id,
        'client_secret': client_secret,
        'scope': scope
    }
    try:
        response = requests.post(url, data=payload)
        response.raise_for_status()
        return response.json().get("access_token")
    except Exception as e:
        print(f"Failed to retrieve SPN token for scope {scope}: {e}")
        return None

# Get Fabric Token (for Get Item API)
fabric_scope = "https://api.fabric.microsoft.com/.default" 
fabric_token = get_spn_token(fabric_scope)

if fabric_token:
    print("Successfully retrieved Fabric token.")
else:
    print("Failed to retrieve Fabric token.")

In [ ]:
# --- Get Item Function ---
def get_fabric_item(workspace_id, item_id):
    """
    Gets details of a specific item from a Fabric Workspace.
    API: GET https://api.fabric.microsoft.com/v1/workspaces/{workspaceId}/items/{itemId}
    """
    if not fabric_token:
        print("Skipping Get Item: No Fabric token available.")
        return None

    url = f"{fabric_api_base}/workspaces/{workspace_id}/items/{item_id}"
    
    headers = {
        "Authorization": f"Bearer {fabric_token}",
        "Content-Type": "application/json"
    }
    
    try:
        response = requests.get(url, headers=headers)
        
        if response.status_code == 200:
            print(f"SUCCESS: Retrieved item {item_id} from workspace {workspace_id}.")
            return response.json()
        elif response.status_code == 404:
            print(f"NOT FOUND: Item {item_id} not found in workspace {workspace_id}.")
            return None
        else:
            print(f"FAILED: Could not get item {item_id}. Status: {response.status_code}, Response: {response.text}")
            return None
            
    except Exception as e:
        print(f"EXCEPTION: Error getting item {item_id}: {e}")
        return None

# --- Main Execution ---
if workspace_id and item_id and workspace_id != "your-workspace-id":
    print(f"Querying Item: {item_id} in Workspace: {workspace_id}")
    item_details = get_fabric_item(workspace_id, item_id)
    
    if item_details:
        print(json.dumps(item_details, indent=4))
else:
    print("Please set valid workspace_id and item_id parameters in the first cell.")